# Caso 2: INTERMEDIO - Calidad de Datos RENAMU 2022 (Registro Nacional de Municipalidades)

---

## Contexto del Proyecto

### Descripción del Problema
El **Instituto Nacional de Estadística e Informática (INEI)**, a través del
**RENAMU** (Registro Nacional de Municipalidades), recopila información
administrativa, de recursos humanos, servicios y gestión de las municipalidades
peruanas. Al consolidar los datos del año 2022, se detectan valores faltantes,
inconsistencias en códigos geográficos y registros duplicados que impiden
generar análisis confiables por municipalidad.

### Objetivo Analítico
Limpiar y mejorar la calidad de los datos del RENAMU 2022 para:
- Imputar valores faltantes en columnas geográficas clave
- Reconstruir Ubigeos incorrectos desde ccdd+ccpp+ccdi
- Corregir inconsistencias entre campos geográficos
- Preparar el dataset Silver listo para el JOIN con SIAF y SISMEPRE

### Impacto de la Mala Calidad de Datos
- **Financiero**: Un Ubigeo incorrecto rompe el JOIN con SIAF y SISMEPRE
- **Operativo**: Duplicados por municipalidad generan conteos incorrectos
- **Estratégico**: Decisiones de política pública basadas en datos
  incompletos afectan la asignación de recursos a municipios

---

## Dimensiones de Calidad a Corregir

1. **Completitud**: Imputar nulos en Departamento, Provincia y Distrito
2. **Exactitud**: Reconstruir Ubigeo desde ccdd+ccpp+ccdi cuando sea inválido
3. **Consistencia**: Corregir inconsistencias entre Ubigeo y códigos geográficos
4. **Integridad**: Validar que ccdd esté en rango 01-25
5. **Razonabilidad**: Corregir Tipomuni fuera de rango 1-5
6. **Oportunidad**: Verificar que todos los registros sean del año 2022
7. **Unicidad**: Eliminar duplicados exactos y por Ubigeo
8. **Validez**: Estandarizar formato del Ubigeo a 6 dígitos



---
# SOLUCIÓN NIVEL INTERMEDIO - RENAMU 2022

## Objetivo
Recuperar datos cuando sea posible usando:
- Reconstrucción del Ubigeo desde ccdd+ccpp+ccdi
- Imputación de nombres geográficos faltantes
- Corrección de inconsistencias entre campos relacionados
- Validaciones basadas en el catálogo geográfico del INEI

### Filosofía:
"No eliminar sin antes intentar recuperar información valiosa —
un Ubigeo con formato inválido puede reconstruirse desde
los códigos ccdd, ccpp y ccdi si estos están correctos"

In [1]:
# ============================================
# INSTALACIÓN Y CARGA DE LIBRERÍAS
# ============================================

# !pip install pandas numpy matplotlib seaborn pyarrow -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ============================================
# CONFIGURACIÓN DE VISUALIZACIÓN
# ============================================

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Mostrar gráficos en notebook
%matplotlib inline

# Configuración opcional de pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.max_colwidth', 100)

print("✅ Librerías cargadas correctamente")

✅ Librerías cargadas correctamente


In [2]:
# ============================================
# CARGA DE DATOS RENAMU 2022
# ============================================

# Leer dataset
df = pd.read_csv(
    'Base_RENAMU_2022_f.csv',
    encoding='latin-1',
    sep=';'
)

# ============================================
# CONFIGURACIÓN INICIAL
# ============================================

# Convertir Ubigeo a string y completar con ceros
df['Ubigeo'] = df['Ubigeo'].astype(str).str.zfill(6)

# Clave de negocio para control de duplicados
cols_duplicados = ['Ubigeo']

# ============================================
# INFORMACIÓN GENERAL DEL DATASET
# ============================================

print("=" * 60)
print("          CARGA DE DATOS RENAMU 2022")
print("=" * 60)

print(f"\n✅ Dataset cargado correctamente")
print(f"\n📊 Registros : {len(df):,}")
print(f"📊 Columnas  : {len(df.columns)}")
print(f"💾 Memoria utilizada : "
      f"{df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n📌 Primeros registros:")
display(df.head())

print("\n📌 Tipos de datos:")
print(df.dtypes)

          CARGA DE DATOS RENAMU 2022

✅ Dataset cargado correctamente

📊 Registros : 1,874
📊 Columnas  : 1368
💾 Memoria utilizada : 22.80 MB

📌 Primeros registros:


,ï»¿AÃ±o,idmunici,ccdd,ccpp,ccdi,Ubigeo,Departamento,Provincia,Distrito,Tipomuni,VFI,P04_1,P04_2,P04_3,P04_4,P04_5,P04_6,P05_CC,P05_1,P05_2,P05_3,P06_CC,P06,P07,P08,P09,P10_1,P10_2,P10_3,P10_4,VFI_P11A,P11A_1,P11A_1_1,P11A_1_2,P11A_2,P11A_2_1,P11A_2_2,P11A_3,P11A_3_1,P11A_3_2,P11A_4,P11A_4_1,P11A_4_2,P11A_5,P11A_5_1,P11A_5_2,P11A_6,P11A_6_1,P11A_6_2,P11A_7,P11A_7_1,P11A_7_2,P11A_8,P11A_8_1,P11A_8_2,P11A_9,P11A_9_1,P11A_9_2,P11A_10,P11A_10_1,P11A_10_2,P11A_10_O,VFI_P11B,P11B_11,P11B_11_1,P11B_11_2,P11B_12,P11B_12_1,P11B_12_2,P11B_13,P11B_13_1,P11B_13_2,P11B_14,P11B_14_1,P11B_14_2,P11B_15,P11B_15_1,P11B_15_2,P11B_16,P11B_16_1,P11B_16_2,P11B_17,P11B_17_1,P11B_17_2,P11B_18,P11B_18_1,P11B_18_2,P11B_18_O,VFI_P12,P12_1,P12A_1,P12_2,P12A_2,VFI_P13,P13_1,P13A_1,P13_2,P13A_2,P13_3,P13A_3,P13_4,P13A_4,P13_5,P13A_5,P13_6,P13A_6,P13_7,P13A_7,P13_8,P13A_8,P13_9,P13A_9,P13A_9_O,VFI_P14,P14,P14A_1,P14A_2,VFI_P15,P15_1,P15A_1,P15_2,P15A_2,P15_3,P15A_3,P15_4,P15A_4,P15_5,P15A_5,P15_6,P15A_6,P15A_6_O,VFI_P16,P16_1,P16_2,P16_3,P16_4,P16_5,P16_6,P16_7,P16_8,P16_9,P16_9_O,VFI_P17,P17_1,P17_2,P17_3,P17_4,P17_5,P17_6,P17_7,P17_8,P17_9,P17_10,P17_11,P17_12,P17_13,P17_13_O,P17_14,VFI_P18,P18,P18_Portal,VFI_P19,P19D_T,P19D_NM,P19D_NH,P19D_CM,P19D_CH,P19D_LM,P19D_LH,P19D_VM,P19D_VH,P19_1_T,P19_1_NM,P19_1_NH,P19_1_CM,P19_1_CH,P19_1_LM,P19_1_LH,P19_1_VM,P19_1_VH,P19_2_T,P19_2_NM,P19_2_NH,P19_2_CM,P19_2_CH,P19_2_LM,P19_2_LH,P19_2_VM,P19_2_VH,P19_3_T,P19_3_NM,P19_3_NH,P19_3_CM,P19_3_CH,P19_3_LM,P19_3_LH,P19_3_VM,P19_3_VH,P19_4_T,P19_4_NM,P19_4_NH,P19_4_CM,P19_4_CH,P19_4_LM,P19_4_LH,P19_4_VM,P19_4_VH,P19_5_T,P19_5_NM,P19_5_NH,P19_5_CM,P19_5_CH,P19_5_LM,P19_5_LH,P19_5_VM,P19_5_VH,P19_6_T,P19_6_NM,P19_6_NH,P19_6_CM,P19_6_CH,P19_6_LM,P19_6_LH,P19_6_VM,P19_6_VH,P19_7_T,P19_7_NM,P19_7_NH,P19_7_CM,P19_7_CH,P19_7_LM,P19_7_LH,P19_7_VM,P19_7_VH,P19M_T,P19M_NM,P19M_NH,P19M_CM,P19M_CH,P19M_LM,P19M_LH,P19M_VM,P19M_VH,VFI_P19A,P19A,P19A_1_T,P19A_1_M,P19A_1_H,P19A_2_T,P19A_2_M,P19A_2_H,VFI_P20,P20,P20_1_T,P20_1_M,P20_1_H,P20_2_T,P20_2_M,P20_2_H,VFI_P21,P21,P21_1_T,P21_1_M,P21_1_H,P21_1_T_20,P21_1_M_20,P21_1_H_20,P21_2_T,P21_2_M,P21_2_H,P21_2_T_20,P21_2_M_20,P21_2_H_20,VFI_P22,P22_AT1,P22_AT2,P22_AT3,P22_AT4,P22_AT5,P22_AT6,P22_AT7,P22_AT8,P22_AT9,P22_AT10,P22_AT11,P22_AT12,P22_AT13,P22_AT14,P22_AT15,P22_AT16,P22_AT17,P22_AT18,P22_AT19,P22_AT20,P22_AT21,P22_AT22,P22_AT23,P22_AT24,P22_AT24_O,P22_C1,P22_C2,P22_C3,P22_C4,P22_C5,P22_C6,P22_C7,P22_C8,P22_C9,P22_C10,P22_C11,P22_C12,P22_C13,P22_C14,P22_C15,P22_C16,P22_C17,P22_C18,P22_C19,P22_C20,P22_C21,P22_C22,P22_C23,P22_C24,P22_C24_O,VFI_P23,P23_1,P23_1_1,P23_1_2,P23_1_3,P23_1_4,P23_1_4_O,P23_2,P23_2_1,P23_2_2,P23_2_3,P23_2_4,P23_2_4_O,P23_3,P23_3_1,P23_3_2,P23_3_4,P23_3_4_O,P23_4,P23_4_1,P23_4_2,P23_4_3,P23_4_4,P23_4_4_O,P23_5,P23_5_1,P23_5_2,P23_5_3,P23_5_4,P23_5_4_O,P23_6,P23_6_1,P23_6_2,P23_6_3,P23_6_4,P23_6_4_O,P23_7,P23_7_1,P23_7_2,P23_7_3,P23_7_4,P23_7_4_O,P23_8,P23_8_1,P23_8_2,P23_8_3,P23_8_4,P23_8_4_O,P23_9,P23_9_1,P23_9_2,P23_9_3,P23_9_4,P23_9_4_O,P23_10,P23_10_1,P23_10_2,P23_10_3,P23_10_4,P23_10_4_O,P23_11,P23_11_1,P23_11_2,P23_11_4,P23_11_4_O,P23_12,P23_12_1,P23_12_2,P23_12_4,P23_12_4_O,P23_13,P23_13_1,P23_13_2,P23_13_4,P23_13_4_O,P23_15,P23_15_1,P23_15_2,P23_15_4,P23_15_4_O,P23_16,P23_16_1,P23_16_2,P23_16_3,P23_16_4,P23_16_4_O,P23_14,P23_14_1_O,P23_14_1,P23_14_2,P23_14_3,VFI_P24,P24,P24_1_T,P24_1_M,P24_1_H,P24_2_T,P24_2_M,P24_2_H,P24_3_T,P24_3_M,P24_3_H,P24_4_T,P24_4_M,P24_4_H,VFI_P25,P25_1,P25_1_1,P25_2,P25_3,P25_4,P25_4_O,P25_5,VFI_P26A,P26A,VFI_P26,P26,P26A_1,VFI_P27,P27,VFI_P28,P28_1,P28_2,P28_3,P28_4,P28_5,P28_5_O,VFI_P29,P29_1,P29_1_1,P29_1_2,P29_1_3,P29_1_4,P29_1_5,P29_1_6,P29_2,P29_2_1,P29_2_2,P29_2_3,P29_2_4,P29_2_5,P29_2_6,P29_3,P29_3_1,P29_3_2,P29_3_3,P29_3_4,P29_3_5,P29_3_6,P29_4,P29_4_1,P29_4_2,P29_4_3,P29_4_4,P29_4_5,P29_4_6,P29_5,P29_5_1,P29_5_2,P29_5_3,P29_5_4,P29_5_5,P29_5_6,P29_6,P29_6_1,P29_6_2,P29_6_3,P29_6_4,P29_6_5,P29_6_6,P29_7,P29_7_1,P29_7_2,P29_7_3,P29_7_4,P29_7_5,P29_7_6,P29_8,P29_8_1,P29


📌 Tipos de datos:
ï»¿AÃ±o          int64
idmunici         int64
ccdd             int64
ccpp             int64
ccdi             int64
                 ...  
C97_2_2_3_11       str
C97_2_2_4          str
C97_2_3            str
C97_2_4         object
C97_3              str
Length: 1368, dtype: object


In [6]:
# ============================================
# PIPELINE SILVER INTERMEDIO - RENAMU 2022
# ============================================

# Crear copia de trabajo
df_intermedio = df.copy()

print("=" * 60)
print("         PIPELINE SILVER INTERMEDIO")
print("=" * 60)

print(f"\n📊 Registros iniciales: {len(df_intermedio):,}")

# ============================================
# DETECCIÓN DE COLUMNA AÑO
# ============================================

# Detectar columna año corregida por problemas de codificación
col_anio = [c for c in df.columns if 'a' in c.lower() and ('ñ' in c.lower() or 'ã' in c.lower())]

if len(col_anio) > 0:
    col_anio = col_anio[0]
else:
    raise ValueError("❌ No se encontró columna de año")

print(f"✅ Columna año detectada: '{col_anio}'")

# ============================================
# NORMALIZACIÓN INICIAL
# ============================================

# UBIGEO como string de 6 dígitos
df_intermedio['Ubigeo'] = (
    df_intermedio['Ubigeo']
    .astype(str)
    .str.zfill(6)
)

# ============================================
# 1. IMPUTACIÓN DE VALORES FALTANTES
# ============================================

print("\n" + "=" * 60)
print("1. IMPUTACIÓN DE VALORES FALTANTES")
print("=" * 60)

columnas_geo = ['Departamento', 'Provincia', 'Distrito']

for col in columnas_geo:

    nulos = df_intermedio[col].isnull().sum()

    if nulos > 0:
        moda = df_intermedio[col].mode()[0]

        df_intermedio[col] = (
            df_intermedio[col]
            .fillna(moda)
        )

    print(f"✅ {col}: {nulos:,} valores imputados")

# ============================================
# 2. CORRECCIÓN DE VALORES INCORRECTOS
# ============================================

print("\n" + "=" * 60)
print("2. CORRECCIÓN DE VALORES INCORRECTOS")
print("=" * 60)

# Reconstrucción de UBIGEO
df_intermedio['Ubigeo_calculado'] = (
    df_intermedio['ccdd'].astype(str).str.zfill(2) +
    df_intermedio['ccpp'].astype(str).str.zfill(2) +
    df_intermedio['ccdi'].astype(str).str.zfill(2)
)

# Corregir solo UBIGEO inválidos
mask_ubigeo = ~df_intermedio['Ubigeo'].str.match(r'^\d{6}$')

cantidad_ubigeo = mask_ubigeo.sum()

df_intermedio.loc[
    mask_ubigeo,
    'Ubigeo'
] = df_intermedio.loc[
    mask_ubigeo,
    'Ubigeo_calculado'
]

print(f"✅ Ubigeo inválido corregido: {cantidad_ubigeo:,}")

# Validar Tipomuni
df_intermedio['Tipomuni'] = pd.to_numeric(
    df_intermedio['Tipomuni'],
    errors='coerce'
)

mask_tipomuni = ~df_intermedio['Tipomuni'].isin([1, 2, 3, 4, 5])

cantidad_tipomuni = mask_tipomuni.sum()

if cantidad_tipomuni > 0:

    moda_tipomuni = (
        df_intermedio.loc[
            ~mask_tipomuni,
            'Tipomuni'
        ]
        .mode()[0]
    )

    df_intermedio.loc[
        mask_tipomuni,
        'Tipomuni'
    ] = moda_tipomuni

# Convertir a entero
df_intermedio['Tipomuni'] = (
    df_intermedio['Tipomuni']
    .astype(int)
)

print(f"✅ Tipomuni inválido corregido: {cantidad_tipomuni:,}")

# ============================================
# 3. ESTANDARIZACIÓN DE FORMATOS
# ============================================

print("\n" + "=" * 60)
print("3. ESTANDARIZACIÓN DE FORMATOS")
print("=" * 60)

for col in columnas_geo:

    df_intermedio[col] = (
        df_intermedio[col]
        .astype(str)
        .str.upper()
        .str.strip()
    )

print("✅ Variables geográficas estandarizadas")

# ============================================
# 4. CORRECCIÓN DE INCONSISTENCIAS
# ============================================

print("\n" + "=" * 60)
print("4. CORRECCIÓN DE INCONSISTENCIAS")
print("=" * 60)

# Detectar inconsistencias UBIGEO
mask_inconsistente = (
    df_intermedio['Ubigeo'] !=
    df_intermedio['Ubigeo_calculado']
)

cantidad_inconsistente = mask_inconsistente.sum()

print(f"✅ Ubigeos inconsistentes detectados: "
      f"{cantidad_inconsistente:,}")

# Eliminar columna auxiliar
df_intermedio = df_intermedio.drop(
    columns=['Ubigeo_calculado']
)

# ============================================
# 5. MANEJO DE DUPLICADOS
# ============================================

print("\n" + "=" * 60)
print("5. MANEJO DE DUPLICADOS")
print("=" * 60)

# Duplicados exactos
antes_dup = len(df_intermedio)

df_intermedio = df_intermedio.drop_duplicates(
    keep='first'
)

dup_exactos = antes_dup - len(df_intermedio)

print(f"✅ Duplicados exactos eliminados: "
      f"{dup_exactos:,}")

# Duplicados por UBIGEO
antes_ubigeo = len(df_intermedio)

df_intermedio = df_intermedio.drop_duplicates(
    subset=cols_duplicados,
    keep='first'
)

dup_ubigeo = antes_ubigeo - len(df_intermedio)

print(f"✅ Duplicados por Ubigeo eliminados: "
      f"{dup_ubigeo:,}")

# ============================================
# 6. VALIDACIÓN FINAL
# ============================================

print("\n" + "=" * 60)
print("6. VALIDACIÓN FINAL")
print("=" * 60)

# Validar ccdd
antes_ccdd = len(df_intermedio)

df_intermedio = df_intermedio[
    (df_intermedio['ccdd'] >= 1) &
    (df_intermedio['ccdd'] <= 25)
]

ccdd_invalidos = antes_ccdd - len(df_intermedio)

print(f"✅ ccdd fuera de rango eliminados: "
      f"{ccdd_invalidos:,}")

# Validar año
antes_anio = len(df_intermedio)

df_intermedio = df_intermedio[
    df_intermedio[col_anio] == 2022
]

anio_invalidos = antes_anio - len(df_intermedio)

print(f"✅ Registros fuera del año 2022 eliminados: "
      f"{anio_invalidos:,}")

# ============================================
# RESUMEN FINAL
# ============================================

perdida_total = len(df) - len(df_intermedio)

print("\n" + "=" * 60)
print("               RESUMEN FINAL")
print("=" * 60)

print(f"\n📊 Registros iniciales : {len(df):,}")
print(f"📊 Registros finales   : {len(df_intermedio):,}")
print(f"📉 Pérdida total       : {perdida_total:,} "
      f"registros ({(perdida_total / len(df)) * 100:.2f}%)")

print("\n✅ Pipeline Silver intermedio completado")

         PIPELINE SILVER INTERMEDIO

📊 Registros iniciales: 1,874
✅ Columna año detectada: 'ï»¿AÃ±o'

1. IMPUTACIÓN DE VALORES FALTANTES
✅ Departamento: 0 valores imputados
✅ Provincia: 0 valores imputados
✅ Distrito: 0 valores imputados

2. CORRECCIÓN DE VALORES INCORRECTOS
✅ Ubigeo inválido corregido: 0
✅ Tipomuni inválido corregido: 0

3. ESTANDARIZACIÓN DE FORMATOS
✅ Variables geográficas estandarizadas

4. CORRECCIÓN DE INCONSISTENCIAS
✅ Ubigeos inconsistentes detectados: 0

5. MANEJO DE DUPLICADOS
✅ Duplicados exactos eliminados: 0
✅ Duplicados por Ubigeo eliminados: 0

6. VALIDACIÓN FINAL
✅ ccdd fuera de rango eliminados: 0
✅ Registros fuera del año 2022 eliminados: 0

               RESUMEN FINAL

📊 Registros iniciales : 1,874
📊 Registros finales   : 1,874
📉 Pérdida total       : 0 registros (0.00%)

✅ Pipeline Silver intermedio completado


In [7]:
# ============================================
# VERIFICACIÓN POST-LIMPIEZA
# ============================================

print("\n" + "=" * 60)
print("        VERIFICACIÓN POST-LIMPIEZA")
print("              RENAMU 2022")
print("=" * 60)

# ============================================
# VALIDACIONES PRINCIPALES
# ============================================

nulos_clave = (
    df_intermedio[
        ['Ubigeo', 'Departamento', 'Provincia', 'Distrito']
    ]
    .isnull()
    .sum()
    .sum()
)

duplicados_ubigeo = (
    df_intermedio
    .duplicated(subset=['Ubigeo'])
    .sum()
)

ubigeo_invalido = len(
    df_intermedio[
        ~df_intermedio['Ubigeo']
        .astype(str)
        .str.match(r'^\d{6}$')
    ]
)

tipomuni_invalido = len(
    df_intermedio[
        ~df_intermedio['Tipomuni']
        .isin([1, 2, 3, 4, 5])
    ]
)

anio_invalido = len(
    df_intermedio[
        df_intermedio[col_anio] != 2022
    ]
)

# ============================================
# RESULTADOS
# ============================================

print(f"\n📊 Total registros                 : "
      f"{len(df_intermedio):,}")

print(f"✅ Valores nulos columnas clave    : "
      f"{nulos_clave:,}")

print(f"✅ Duplicados por Ubigeo           : "
      f"{duplicados_ubigeo:,}")

print(f"✅ Ubigeo inválido                 : "
      f"{ubigeo_invalido:,}")

print(f"✅ Tipomuni fuera de rango         : "
      f"{tipomuni_invalido:,}")

print(f"✅ Registros fuera del 2022        : "
      f"{anio_invalido:,}")

print("\n" + "-" * 60)

print(f"📍 Departamentos únicos            : "
      f"{df_intermedio['Departamento'].nunique():,}")

print(f"📍 Provincias únicas               : "
      f"{df_intermedio['Provincia'].nunique():,}")

print(f"📍 Distritos únicos                : "
      f"{df_intermedio['Distrito'].nunique():,}")

print(f"📍 Ubigeos únicos                  : "
      f"{df_intermedio['Ubigeo'].nunique():,}")

print("\n" + "-" * 60)

print("📌 Valores únicos Tipomuni:")
print(sorted(df_intermedio['Tipomuni'].unique()))

print("\n✅ Verificación completada correctamente")


        VERIFICACIÓN POST-LIMPIEZA
              RENAMU 2022

📊 Total registros                 : 1,874
✅ Valores nulos columnas clave    : 0
✅ Duplicados por Ubigeo           : 0
✅ Ubigeo inválido                 : 0
✅ Tipomuni fuera de rango         : 0
✅ Registros fuera del 2022        : 0

------------------------------------------------------------
📍 Departamentos únicos            : 25
📍 Provincias únicas               : 196
📍 Distritos únicos                : 1,722
📍 Ubigeos únicos                  : 1,874

------------------------------------------------------------
📌 Valores únicos Tipomuni:
[np.int64(1), np.int64(2)]

✅ Verificación completada correctamente


In [9]:
# ============================================
# EXPORTACIÓN A PARQUET
# ============================================

print("\nEXPORTACIÓN DATASET SILVER INTERMEDIO\n")

# Convertir columnas tipo object a string
cols_object = df_intermedio.select_dtypes(include='object').columns

for col in cols_object:
    df_intermedio[col] = df_intermedio[col].astype(str)

# Exportar archivo parquet
df_intermedio.to_parquet(
    "renamu_silver_intermedio.parquet",
    index=False
)

# Resumen exportación
print("✅ Archivo exportado correctamente")
print(f"📁 Nombre archivo : renamu_silver_intermedio.parquet")
print(f"📊 Registros      : {len(df_intermedio):,}")
print(f"📋 Columnas       : {len(df_intermedio.columns)}")
print(f"💾 Memoria usada  : {df_intermedio.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n✅ Dataset Silver Intermedio listo para análisis")


EXPORTACIÓN DATASET SILVER INTERMEDIO

✅ Archivo exportado correctamente
📁 Nombre archivo : renamu_silver_intermedio.parquet
📊 Registros      : 1,874
📋 Columnas       : 1368
💾 Memoria usada  : 22.26 MB

✅ Dataset Silver Intermedio listo para análisis


### Conclusiones de la Solución Intermedia

**Ventajas:**
- Se verificó y estandarizó correctamente el campo Ubigeo, garantizando un formato válido de 6 dígitos.
- Se mantuvo el 100% de los registros originales, evitando pérdida de información durante la limpieza.
- Se eliminaron inconsistencias potenciales entre Ubigeo y los códigos geográficos
- El dataset quedó listo para integraciones y análisis posteriores con otras fuentes como SIAF y SISMEPRE.

**Desventajas:**
- La imputación por moda permanece limitada como estrategia general, aunque en este dataset no fue necesaria debido a la ausencia de valores nulos.
- No se identifican casos donde un Ubigeo tenga estructura válida pero represente un distrito incorrecto.
- No se detectaron errores reales en los datos, por lo que varias reglas de limpieza no tuvieron impacto práctico.